# Rimozione delle traiettorie expert con collisione

Il notebook legge `expert_trajectories.pkl`, rimuove ogni traiettoria che contiene almeno una transizione con status `collided` e salva il risultato in `expert_trajectories_no_collision.pkl`.

Il file originale viene aperto esclusivamente in lettura. L'output non viene sovrascritto se esiste gia'. Al termine vengono ricaricati e verificati entrambi i file.

In [ ]:
import hashlib
import pickle
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path.cwd()
if not (DATA_DIR / "expert_trajectories.pkl").exists():
    candidate = Path.cwd() / "datasets"
    if (candidate / "expert_trajectories.pkl").exists():
        DATA_DIR = candidate

SOURCE_PATH = DATA_DIR / "expert_trajectories.pkl"
OUTPUT_PATH = DATA_DIR / "expert_trajectories_no_collision.pkl"
COLLIDED_STATUS_INDEX = 1

print("Input :", SOURCE_PATH.resolve())
print("Output:", OUTPUT_PATH.resolve())

## 1. Caricamento in sola lettura

In [ ]:
if not SOURCE_PATH.is_file():
    raise FileNotFoundError(f"File non trovato: {SOURCE_PATH.resolve()}")
if SOURCE_PATH.resolve() == OUTPUT_PATH.resolve():
    raise ValueError("Input e output devono essere file differenti.")

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

source_hash_before = sha256_file(SOURCE_PATH)
with SOURCE_PATH.open("rb") as stream:
    expert_trajectories = pickle.load(stream)

if not isinstance(expert_trajectories, list):
    raise TypeError(f"Attesa una list, trovato {type(expert_trajectories).__name__}")
if not expert_trajectories:
    raise ValueError("Il dataset expert e' vuoto.")

print(f"Caricate {len(expert_trajectories):,} traiettorie")
print(f"SHA-256 originale: {source_hash_before}")

## 2. Validazione degli status e filtraggio

Una traiettoria viene rimossa se `next_status[1] == 1` in almeno una sua transizione. Questo controllo e' piu' robusto del solo status terminale.

In [ ]:
STATUS_NAMES = [
    "arrived", "collided", "off_road", "timeout",
    "running", "teleported", "removed_unknown",
]

def validated_status(transition, trajectory_index, transition_index):
    status = np.asarray(transition.next_status)
    if status.shape != (7,):
        raise ValueError(
            f"Status con shape {status.shape} nella traiettoria "
            f"{trajectory_index}, transizione {transition_index}"
        )
    if not np.isfinite(status).all():
        raise ValueError(
            f"Status non finito nella traiettoria {trajectory_index}, "
            f"transizione {transition_index}"
        )
    if not np.all(np.isin(status, [0, 1])) or not np.isclose(status.sum(), 1):
        raise ValueError(
            f"Status non one-hot nella traiettoria {trajectory_index}, "
            f"transizione {transition_index}: {status}"
        )
    return status

def trajectory_has_collision(trajectory, trajectory_index):
    if len(trajectory) == 0:
        raise ValueError(f"Traiettoria vuota all'indice {trajectory_index}")
    return any(
        validated_status(transition, trajectory_index, transition_index)[COLLIDED_STATUS_INDEX] == 1
        for transition_index, transition in enumerate(trajectory)
    )

collision_mask = [
    trajectory_has_collision(trajectory, trajectory_index)
    for trajectory_index, trajectory in enumerate(expert_trajectories)
]

filtered_trajectories = [
    trajectory
    for trajectory, has_collision in zip(expert_trajectories, collision_mask)
    if not has_collision
]
removed_trajectories = [
    trajectory
    for trajectory, has_collision in zip(expert_trajectories, collision_mask)
    if has_collision
]

summary = pd.Series({
    "input_trajectories": len(expert_trajectories),
    "removed_collision_trajectories": len(removed_trajectories),
    "output_trajectories": len(filtered_trajectories),
    "removed_fraction": len(removed_trajectories) / len(expert_trajectories),
    "input_transitions": sum(map(len, expert_trajectories)),
    "removed_transitions": sum(map(len, removed_trajectories)),
    "output_transitions": sum(map(len, filtered_trajectories)),
})
display(summary.to_frame("value"))

## 3. Controllo degli esiti terminali

In [ ]:
def terminal_outcome(trajectory):
    status = np.asarray(trajectory[-1].next_status)
    return STATUS_NAMES[int(np.argmax(status))]

outcome_table = pd.DataFrame({
    "original": pd.Series(Counter(map(terminal_outcome, expert_trajectories))),
    "filtered": pd.Series(Counter(map(terminal_outcome, filtered_trajectories))),
}).fillna(0).astype(int)
display(outcome_table)

if any(trajectory_has_collision(trajectory, index)
       for index, trajectory in enumerate(filtered_trajectories)):
    raise AssertionError("Il dataset filtrato contiene ancora collisioni.")

print("Controllo superato: nessuna collisione nel dataset filtrato.")

## 4. Salvataggio sicuro

La modalita' `xb` crea un file nuovo e genera un errore se il percorso esiste gia'. In questo modo un'esecuzione accidentale non sovrascrive output precedenti.

In [ ]:
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"L'output esiste gia' e non verra' sovrascritto: {OUTPUT_PATH.resolve()}"
    )

with OUTPUT_PATH.open("xb") as stream:
    pickle.dump(filtered_trajectories, stream, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Salvato: {OUTPUT_PATH.resolve()}")
print(f"Dimensione: {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MiB")

## 5. Verifica finale

Il file generato viene ricaricato; vengono controllati conteggio, assenza di collisioni e hash del file originale.

In [ ]:
with OUTPUT_PATH.open("rb") as stream:
    reloaded_trajectories = pickle.load(stream)

assert len(reloaded_trajectories) == len(filtered_trajectories)
assert not any(
    trajectory_has_collision(trajectory, index)
    for index, trajectory in enumerate(reloaded_trajectories)
)

source_hash_after = sha256_file(SOURCE_PATH)
assert source_hash_after == source_hash_before, "Il file originale e' cambiato."

verification = pd.Series({
    "output_exists": OUTPUT_PATH.is_file(),
    "output_trajectories": len(reloaded_trajectories),
    "output_contains_collisions": False,
    "source_sha256_before": source_hash_before,
    "source_sha256_after": source_hash_after,
    "source_unchanged": source_hash_before == source_hash_after,
})
display(verification.to_frame("value"))
print("Verifica completata con successo. Il file originale non e' stato modificato.")